# BEH 60: analysis datasets and dictionary

The tables Simon models on, inside Fabric (no extract leaves Fabric, so keys are not tokenised: clarifications E5):

- `beh.CustomerMonthDataset`: one row per customer and month-end, with holdings and lifecycle (CustomerMonth), rolling
  behaviour (CustomerMonthBehaviour), Motor renewal history (CustomerMonthRenewalHistory) and outcomes
  (CustomerMonthOutcome).
- `beh.MotorPolicyMonthDataset`: one row per Motor policy and month-end, the policy's lifecycle position and renewal
  outcome with the customer's readings beside it: the grain for "is behaviour at M3, M6 or M11 associated with
  renewal?".
- `beh.Coverage` (each source family's coverage start) and `beh.Dictionary` (what each column means).

Filter on `IsAnalysisMonth = 1`. For retention use `IsLifecycleObservation = 1` and `RenewalOutcomeObservable = 1`.
Columns marked OUTCOME in the dictionary are after the month-end and are never inputs.

**If a cell fails**

1. Read the description above the failing cell: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_60_Dataset", "started")


## Coverage


**`beh.Coverage`**

Coverage start per source family: a window starting earlier is null in CustomerMonthBehaviour.

- **Reads:** settings cov\_\*, hist_start
- **Key:** CoverageFamily

In [ ]:
create or replace table ${beh.target}.Coverage as
select  *
from    values
        ('events', to_date('${beh.cov_events}'), 'beh.cov_events', 'ads_v2.EpisodeEventsNumbered linked events (ads_v2 window start)'),
        ('quotes', to_date('${beh.cov_quotes}'), 'beh.cov_quotes', 'online quotes and re-quotes linked to a policyholder''s SCV key: GDPR wipe before 2026-05-06'),
        ('calls', to_date('${beh.cov_calls}'), 'beh.cov_calls', 'Genesys calls (CallSession) and the call-derived events: full volume from Nov 2022'),
        ('mychill', to_date('${beh.cov_mychill}'), 'beh.cov_mychill', 'MyChill document workflow events (document friction, chase reminders): from Feb 2024'),
        ('arrears', to_date('${beh.cov_arrears}'), 'beh.cov_arrears', 'arrears events (ARR.*): built in volume from 2023'),
        ('mta_count', to_date('${beh.cov_mta_count}'), 'beh.cov_mta_count', 'ods_v2.PolicyMtaDay.MTAGrossNum (warehouse MTA count per posting day, from 2021-04-01)'),
        ('mta', to_date('${beh.cov_mta}'), 'beh.cov_mta', 'ods_v2.PolicyMta change detail: dlk.rbsdata2_mta_history holds history only from 2026-01-22'),
        ('loan', to_date('${beh.cov_loan}'), 'beh.cov_loan', 'ods_v2.PolicyLoanStatus'),
        ('comms', to_date('${beh.cov_comms}'), 'beh.cov_comms', 'ads_v2.CampaignHistory (Xtremepush email and SMS): full volume from Dec 2022'),
        ('consent', to_date('${beh.cov_consent}'), 'beh.cov_consent', 'ods_v2.ConsentAction (Mercury marketing consent, contact-centre apps CCI EXO / CCI SE) linked through ads_v2.ContactScvLink'),
        ('portal', to_date('${beh.cov_portal}'), 'beh.cov_portal', 'ods_v2.RenewalPortalEvent log-ins linked through ods_v2.ClientPortfolio'),
        ('docwf', to_date('${beh.cov_docwf}'), 'beh.cov_docwf', 'ods_v2.DocumentWorkflowStatus (MyChill document decisions): at volume from 2024'),
        ('mfq', to_date('${beh.cov_mfq}'), 'beh.cov_mfq', 'MFQ online Motor quotes linked to the policyholder (proposer driver id in ods.scveadmlinkfinal, then the retrieval chain). Before May 2026 these are almost only the quotes on the way to buying'),
        ('mfq_device', to_date('${beh.cov_mfq_device}'), 'beh.cov_mfq_device', 'MFQ first and last device per quote (ods_v2.QuoteMotorSummary, the warehouse MFQ fact, from 2024)'),
        ('spine', to_date('${beh.hist_start}'), 'beh.hist_start', 'ods_v2.PolicyMonthEnd holdings and ods_v2.PolicyRenewal due dates'),
        ('web', to_date('${beh.cov_web}'), 'beh.cov_web', 'ods_v2.WebSession (warehouse GA4 clicks from Apr 2023), sessions linked to the customer by the GA cookie through a linked MFQ quote, or by the email the warehouse ties to the cookie (ads_v2.ContactScvLink)'),
        ('tcall', to_date('${beh.cov_tcall}'), 'beh.cov_tcall', 'ods_v2.TelephonyCall inbound calls by queue (queue names from 2022), linked by client code, else by phone number'),
        ('fv3', to_date('${beh.cov_friction_v3}'), 'beh.cov_friction_v3', 'friction extract to John''s spec v3.3 (beh.friction_v3 setting: ods.friction_episode_v3_<data date>), completed episodes from Jul 2022, call-wait measures from Nov 2022')
        as t(CoverageFamily, CoverageStart, Setting, RestsOn)


## CustomerMonthDataset and MotorPolicyMonthDataset


In [ ]:
%%pyspark
# The two analysis tables Simon models on, joined in pyspark so every column is carried without listing it.
# The parts are 1:1 on customer x month-end (each checked unique by its own notebook).
keys = ["SCV_Customer_Key", "MonthEndDate"]
cm = spark.table(f"{target_schema}.CustomerMonth")
parts = [spark.table(f"{target_schema}.{t}") for t in
         ["CustomerMonthBehaviour", "CustomerMonthRenewalHistory", "CustomerMonthPurchase", "CustomerMonthOutcome"]]
ds = cm
for p in parts:
    clash = (set(ds.columns) & set(p.columns)) - set(keys)
    if clash:
        raise Exception(f"Column names repeat across the dataset parts: {sorted(clash)}. Rename one side.")
    ds = ds.join(p, keys, "left")
ds.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_schema}.CustomerMonthDataset")

pm = spark.table(f"{target_schema}.CustomerPolicyMonth").where("PolicyTypeGroup = 'Motor'")
dsp = spark.table(f"{target_schema}.CustomerMonthDataset").drop("MonthNo", "IsAnalysisMonth")
clash = (set(pm.columns) & set(dsp.columns)) - set(keys)
if clash:
    raise Exception(f"Column names repeat between CustomerPolicyMonth and CustomerMonthDataset: {sorted(clash)}.")
pm.join(dsp, keys, "left").write.mode("overwrite").option("overwriteSchema", "true") \
  .saveAsTable(f"{target_schema}.MotorPolicyMonthDataset")
print("CustomerMonthDataset", spark.table(f"{target_schema}.CustomerMonthDataset").count(),
      "MotorPolicyMonthDataset", spark.table(f"{target_schema}.MotorPolicyMonthDataset").count())


## Dictionary


**`beh.Dictionary`**

Column descriptions for the beh tables (generated from build/measures.py and the notebook definitions).

- **Reads:** build definitions
- **Key:** TableName + ColumnName

In [ ]:
create or replace table ${beh.target}.Dictionary as
select  *
from    values
        ('CustomerPolicyMonth', 'SCV_Customer_Key', 'SCV customer key (ods.scveadmlinkfinal, policy link).', '', '', 'Key'),
        ('CustomerPolicyMonth', 'PolicyCode', 'Relay policy code of the in-force term.', '', '', 'Key'),
        ('CustomerPolicyMonth', 'PolicyTypeGroup', 'Motor, Home or Van.', '', '', 'Filter'),
        ('CustomerPolicyMonth', 'MonthEndDate', 'Observation date: the calendar month-end.', '', '', 'Key'),
        ('CustomerPolicyMonth', 'IsAnalysisMonth', '1 from first_obs on; earlier months exist only as history for the 24-month windows.', '', '', 'Filter'),
        ('CustomerPolicyMonth', 'RenewalDate', 'The policy''s renewal date as it stood at the month-end (PolicyMonthEnd.PolicyRenewDateAdj).', '', '', 'Input'),
        ('CustomerPolicyMonth', 'TermStartDate', 'Renewal date minus 12 months.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'MonthsToRenewal', 'Calendar months from the month-end to the renewal month (12 = M1 ... 2 = M11, 1 = M12).', '', '', 'Input'),
        ('CustomerPolicyMonth', 'PolicyMonth', '13 minus MonthsToRenewal: M1 to M11 are the brief''s observation months, M12 is the renewal month. Values below 1 (a renewal date more than 12 months away, e.g. a first term longer than a year) and 13 (renewal date passed but the term still open at the month-end) occur and are not lifecycle observations.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'DaysToRenewal', 'Days from the month-end to the renewal date.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'IsLifecycleObservation', '1 for M1 to M11 (2 to 12 months to renewal): readings usable against this renewal''s outcome.', '', '', 'Filter'),
        ('CustomerPolicyMonth', 'InRenewalWindow', '1 for M12 or later (renewal under way): not an observation for this renewal''s outcome.', '', '', 'Filter'),
        ('CustomerPolicyMonth', 'RenewalOffered', 'OUTCOME. 1 when the upcoming renewal was offered, 0 when its PolicyRenewal row shows no offer (PolicyOfferNum blank), null when no PolicyRenewal row matches the renewal date as it stood (the date changed later).', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalRetained', 'OUTCOME. 1 retained, 0 offered and not retained. Motor: the agreed renewal definition (any of four signals, stg.beh_motor_retention, out/renewal_definition.md); Home and Van: the warehouse flag PolicyRetNum until their lifecycles are built. Null when not offered, and null until the renewal is due by renewal_outcome_end (its outcome has not settled).', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalRetentionSignal', 'OUTCOME. Motor only: which signal retained the upcoming renewal (1 warehouse flag, 2 same policy rolled on, 3 same customer new Motor policy, 4 renewals fact row a year on); null when not retained or not settled.', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalOutcomeObservable', '1 when the renewal is due on or before renewal_outcome_end, so its flags have settled.', '', '', 'Filter'),
        ('CustomerPolicyMonth', 'RenewalSubChannel', 'OUTCOME. Channel the upcoming renewal was processed through (cost to serve, track 2).', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalChannel', 'OUTCOME. Channel group the upcoming renewal was processed through.', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalTYPolicyCode', 'OUTCOME. Policy code of the renewed term (null when not renewed).', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'RenewalPortal', 'OUTCOME. Renewals portal flag of the upcoming renewal.', '', '', 'Outcome'),
        ('CustomerPolicyMonth', 'PolicyId', 'ods_v2 surrogate policy id.', '', '', 'Key'),
        ('CustomerPolicyMonth', 'ClientCode', 'Relay client code of the policyholder.', '', '', 'Key'),
        ('CustomerPolicyMonth', 'MonthNo', 'Month index (year x 12 + month), for window arithmetic.', '', '', 'Key'),
        ('CustomerPolicyMonth', 'PolicyStatusCode', 'Policy status code at the month-end (PolicyMonthEnd).', '', '', 'Input'),
        ('CustomerPolicyMonth', 'PolicyStatusDesc', 'Policy status at the month-end.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'ReportingSaleType', 'How the term was sold (new business or renewal), as the warehouse reports it.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'PolicyInceptDate', 'Inception date of the policy.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'EURGrossPremClose', 'Gross premium of the term in force at the month-end, EUR.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'DiaryPaymentTypeTY', 'Payment type of the term in force. Filled on only about 9% of policy-months (the warehouse sets it on few terms).', '', '', 'Input'),
        ('CustomerPolicyMonth', 'RiskProfileBand', 'Warehouse risk profile band at the month-end.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'VehicleCoverType', 'Motor cover type at the month-end.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'ChannelGroup', 'Channel group the term was sold through.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'ClientAgeBand', 'Policyholder age band at the month-end.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'KeyCareHeld', 'Motor: 1 when the term in force at the month-end carries KeyCare (PolicySale.KeyCareFlag on its latest sale row), 0 when not. Null for Home and Van, and for terms sold before 2023 (the flag is recorded from 2023). Curation #13.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'NcdProtected', 'Motor: 1 when the proposer version valid at the month-end has Full NCD protection, 0 for any other level. Null for Home and Van. Step Back (about 70% of versions) is the standard terms, so it does not count as an add-on. Curation #13.', '', '', 'Input'),
        ('CustomerPolicyMonth', 'NcdProtectionLevel', 'Motor: NCD protection level on the proposer version valid at the month-end (Full, Step Back, Partial, None).', '', '', 'Input'),
        ('CustomerMonth', 'PoliciesHeld', 'In-force Motor, Home and Van policies at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'ProductsHeld', 'Distinct products held (0 to 3).', '', '', 'Input'),
        ('CustomerMonth', 'TenureMonths', 'Months since the customer''s first Motor, Home or Van inception.', '', '', 'Input'),
        ('CustomerMonth', 'TotalPremium', 'Sum of close premium (EUR) of the policies held.', '', '', 'Input'),
        ('CustomerMonth', 'MotorMonthsToRenewal', 'Months to the nearest Motor renewal (null without Motor).', '', '', 'Input'),
        ('CustomerMonth', 'MotorPolicyMonth', '13 minus MotorMonthsToRenewal.', '', '', 'Input'),
        ('CustomerMonth', 'AnyPolicyInRenewalWindow', '1 when any held policy is in its renewal month or later.', '', '', 'Input'),
        ('CustomerMonth', 'MotorPolicies', 'In-force Motor policies at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'HomePolicies', 'In-force Home policies at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'VanPolicies', 'In-force Van policies at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'HoldsMotor', '1 when the customer holds Motor at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'HoldsHome', '1 when the customer holds Home at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'HoldsVan', '1 when the customer holds Van at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'CustomerSinceDate', 'Earliest Motor, Home or Van inception date of the customer.', '', '', 'Input'),
        ('CustomerMonth', 'NextRenewalDate', 'Nearest renewal date of any held policy, as it stood at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'MonthsToNextRenewal', 'Calendar months to the nearest renewal of any held policy.', '', '', 'Input'),
        ('CustomerMonth', 'NextMotorRenewalDate', 'Nearest Motor renewal date, as it stood at the month-end.', '', '', 'Input'),
        ('CustomerMonth', 'MotorKeyCareHeld', '1 when any Motor policy held at the month-end carries KeyCare (null without Motor or without a recorded flag). Curation #13.', '', '', 'Input'),
        ('CustomerMonth', 'MotorNcdProtected', '1 when any Motor policy held at the month-end has Full NCD protection. Curation #13.', '', '', 'Input'),
        ('CustomerMonth', 'MotorAddOnCount', 'Add-on orientation (curation #13): the most add-ons on one Motor policy held at the month-end, 0 to 2 (KeyCare, Full NCD protection). KeyCare is the only add-on product visible in the data.', '', '', 'Input'),
        ('MotorRenewalCycle', 'PromptsBeforeAction', 'Chill prompts in the cycle before the customer''s first renewal action (all prompts if none).', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'ActedBeforeInvite', 'Proactivity: 1 when the customer''s first renewal action came before the first invitation (R1a / R1b). Null when no invitation was recorded.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'DaysInviteToAction', 'Days from the first invitation to the first customer action; negative when the customer acted first.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'RenewalRoute', 'Retained outside the warehouse flag (retained by signal 2 to 4: no processing record) / Online, no call / Online after a call / Assisted, customer phoned / Assisted, no inbound call / Not renewed, customer phoned / Not renewed, online contact only / Not renewed, no contact. Online = processed through e) Online. Null when the cycle window starts before call coverage (EventWindowCovered = 0: renewals due before 30 January 2023).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'ReassuranceCall', 'Renewed cycles: 1 when the customer phoned (R2b) before accepting (before the due date when the accept time is derived). Re-quote condition dropped (GDPR).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'Concession', 'Retained cycles: 1 when the fee paid (TYEURFees) is below the offered fee (RenEURFee) on the renewal''s own PolicyRenewal row; null when not retained. ConcessionFeeEUR is the reduction.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'Requote', '1 when the customer re-quoted online from the renewal price. Null before quote coverage.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'PriceMovement', '(Invite premium - last year''s premium) / last year''s premium. Null without either premium or outside -75% to +300%.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'RenewedAfterPriceRise', 'Retained, for cycles whose invite rose 5% or more; null otherwise.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PaidVsOffered', 'Renewed cycles: (premium paid - offer) / offer. Below zero = paid less than offered.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'ProcessingSubChannel', 'Channel the renewal was processed through.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'RenEURInviteChgIncFee', 'Warehouse definition. Invite price including fee minus last year''s premium including fees, EUR.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'RenEUROfferChgIncFee', 'Warehouse definition. Final offer price including fee minus last year''s premium including fees, EUR.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'GWPYoYCustChg', 'Warehouse definition, customer view (fees included). Price paid minus last year''s when retained; offer minus last year''s when not. EUR. Uses the price paid, so for a renewed cycle it is partly an outcome.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'GWPYoYCustPct', 'Warehouse definition. GWPYoYCustChg / (last year''s premium + fees), held to 2 decimals; 0 when last year''s price is 0. Feeds Chill''s retention report (SL_04).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'GWPYoYInsurerChg', 'Warehouse definition, insurer view (fees excluded). As GWPYoYCustChg without fees. EUR.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'GWPYoYInsurerPct', 'Warehouse definition. GWPYoYInsurerChg / last year''s premium, 2 decimals; 0 when last year''s premium is 0.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PriceIncrease', 'Warehouse band of GWPYoYCustChg in EUR (''a) < -150'' to ''m) 300+''; ''z) NA'' when not calculable).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PriceIncreasePC', 'Warehouse band of GWPYoYCustPct (''a) < -40%'' to ''m) 50%+''). The band Chill''s retention reporting uses.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PriceIncreaseOffer', 'Warehouse band of RenEUROfferChgIncFee in EUR. ''n) Nil Premium'' when there is no offer price.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'DueDate', 'Renewal due date (LYPolicyRenewDateAdj).', '', '', 'Key'),
        ('MotorRenewalCycle', 'DueMonthNo', 'Month index of the due date.', '', '', 'Key'),
        ('MotorRenewalCycle', 'EventWindowCovered', '1 when the event stream and calls cover the whole cycle window (due minus 90 days on). Event-based readings are null otherwise.', '', '', 'Filter'),
        ('MotorRenewalCycle', 'QuoteWindowCovered', '1 when online quotes cover the cycle window (from 6 May 2026). Requote is null otherwise.', '', '', 'Filter'),
        ('MotorRenewalCycle', 'CallWindowCovered', '1 when calls cover the cycle window. InboundCalls is null otherwise.', '', '', 'Filter'),
        ('MotorRenewalCycle', 'Retained', 'OUTCOME of this cycle. 1 retained under the agreed renewal definition (any of four signals: the warehouse flag, the same policy rolled on, the same SCV customer took a Motor policy from 60 days before to 30 days after the due that renews no other policy, or a renewals fact row a year on; out/renewal_definition.md), 0 not.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'RetentionSignal', 'OUTCOME. The first signal that retained the cycle (1 warehouse flag, 2 same policy rolled on, 3 same customer new Motor policy, 4 renewals fact row a year on); null when not retained. Signals 2 to 4 have no processing record on the warehouse fact, so their route reads ''Retained outside the warehouse flag''.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'LYEURGrossPremium', 'Last year''s premium, EUR, without fees.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'RenEURPremInvite', 'Renewal invite premium, EUR, without fees.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'RenEURPremOffer', 'Final offer premium, EUR, without fees.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'TYEURGrossPremium', 'Premium of the renewed term, EUR, without fees (renewed cycles).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PriceRise5', '1 when PriceMovement is 5% or more, 0 when below, null when PriceMovement is null.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'PromoCodeApplied', 'Y when a promo code was applied (recorded from 2025).', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'PromoCodeTreatment', 'Promo code treatment recorded on the renewal.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'ProcessingChannel', 'Channel group the renewal was processed through.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'FirstPromptTime', 'First Chill prompt in the cycle with a usable time.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'FirstInviteTime', 'First invitation (R1a or R1b) in the cycle with a usable time.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'FirstActionTime', 'First customer renewal action in the cycle with a usable time.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'ConcessionAfterContact', 'Retained cycles: 1 when the fee was reduced (Concession = 1) and the customer phoned Chill (R2b, or an inbound call linked by phone) on or before the day they accepted the renewal (the due date when the accept time is derived), so a reduction given on the renewal call itself counts. 0 otherwise. A proxy for a concession the customer asked for: the data has no request flag. Null when not retained or before call coverage. Curation #21.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'ConcessionFeeEUR', 'Fee reduction when Concession = 1 (offered fee minus fee paid), EUR. Median EUR 25 to 34 a year. A few hundred values above EUR 200 (up to about EUR 10,000, mostly 2025 and 2026) pull the mean up: use the median or cap it.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'RenEURFee', 'Fee offered at renewal, EUR.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'TYEURFees', 'Fee paid on the renewed term, EUR.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'FrictionEvents', 'Friction events in the cycle window (stable code set).', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FrictionV3Eligible', 'Friction measures of spec v3.3 the cycle''s Car renewal episode was eligible for (friction extract v3.3, matched on policy code and due date; run note section 12). Null when no episode matched, before cov_friction_v3, or while the episode was open at the extract''s data date.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FrictionV3Affected', 'Of those, the measures where the customer met the friction (spec v3.3 Affected).', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FrictionV3Rate', 'FrictionV3Affected / FrictionV3Eligible: the renewal''s friction rate to spec v3.3, comparable across renewals with different opportunities. Null without an eligible measure. Recorded during the renewal: a Cycle reading, an input only as LR_FrictionV3Rate.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FrictionV3ResolvedShare', 'Share of the renewal''s affected v3.3 friction measures with a known resolution that were resolved. Null without one.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FailedPayments', 'Failed payment events in the cycle window.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'FailedLogins', 'Failed portal log-ins in the cycle window.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'InboundCalls', 'Inbound voice conversations in the cycle window (due minus 90 to plus 30 days), from any number linked to the customer.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'CycleEvents', 'Events attached to the cycle.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'AmbiguousCycleEvents', 'Events that fell in two cycles'' windows (customers with several Motor policies), attached to the nearest.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'DisruptionClaim', '1 when a claim declared on the policy is dated in the term, up to the invitation (due minus 60 days) (PolicyClaimMotor). Declared claims history: a claim may appear only at the next change. Curation #50.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'DisruptionArrears', '1 when the customer had an arrears step in the term, up to the invitation (due minus 60 days). Null before arrears coverage. Curation #50.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'DisruptionPriceRise', '1 when the invite price rose 5% or more (as PriceRise5). Curation #50.', '', '', 'Known at invite'),
        ('MotorRenewalCycle', 'OutboundCalls', 'Outbound voice conversations to the customer in the cycle window (due minus 90 to plus 30 days), any queue except QD_. Null before call coverage.', '', '', 'Cycle reading'),
        ('MotorRenewalCycle', 'RenewedOnOfferNoContact', 'Curation #1, renewed on the offer without help: 1 when the offer was sent (first R1a or R1b invitation), the customer''s first portal log-in in the cycle came after it, they paid in the renewal portal on the policy (due minus 60 to plus 30 days), the renewal was processed online (e) Online) and there was no inbound or outbound call in the cycle window. 0 otherwise. Null when no invitation was recorded (about 1% of cycles) or before event coverage. ''Used the link in the offer'' cannot be seen: the portal does not record how the customer arrived.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'RenewedOnOfferNoContactNoQuote', 'RenewedOnOfferNoContact with no online quote or re-quote in the cycle as well. Set only where online quotes cover the cycle window (from 6 May 2026, QuoteWindowCovered = 1); the quote condition removes about 5% of the renewals that pass the rest.', '', '', 'Outcome'),
        ('MotorRenewalCycle', 'Disrupted', '1 when any of DisruptionClaim, DisruptionArrears, DisruptionPriceRise. Read Retained against it for outcome after disruption (curation #50). Known at the invitation, so usable as a renewal-level input; the curation reads it as an outcome analysis rather than a segmentation dimension.', '', '', 'Known at invite'),
        ('CustomerMonthPurchase', 'MfqPurchases', 'MFQ online purchases of a Motor policy by the customer from January 2024 to the month-end (ods_v2.Quote ConvertedToPolicy).', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_PurchaseDate', 'Date of the customer''s latest MFQ purchase made by the month-end. LP_ columns describe that purchase; all are null for customers who have not bought through MFQ since January 2024.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_PredecessorQuotes', 'Quotes in the purchase''s MFQ retrieval chain made in the 90 days before it.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_StandardConfig', 'Curation #14: 0 when a predecessor quote differs from the purchased quote in cover (the cover chosen at the selection step where both quotes have one, else the cover shown), NCD protection shown, voluntary excess or NCD years; 1 otherwise, including buyers with no predecessor. A customer who started a fresh quote instead of retrieving one is not linked, so 0 is a lower bound.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_ProtectionChange', 'Curation #16: added, removed, both or no change, comparing the purchased quote with its predecessors on KeyCare selected and the NCD protection shown (Step Back and FBP against Step Back Only).', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_KeyCareSelected', '1 when KeyCare was selected on the purchased quote (any insurer response).', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_KeyCareBought', '1 when the policy bought carries KeyCare on its new-business sale (PolicySale.KeyCareFlag). KeyCare is selected on 4% to 5% of purchased quotes but bought on 9% to 11% of sales: it is often added after the quote.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_EntryChannel', 'Curation #15: entry channel of the purchased quote, grouped generic (PPC Generic, SEO Generic), branded (PPC Brand, SEO Brand), direct and other, or unknown (Chill''s own QuoteMotorSummary.FirstMktChannel). A retrieved quote carries its first quote''s channel.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_FirstMktChannel', 'Chill''s entry channel of the purchased quote, ungrouped.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_ReturnGapDays', 'Curation #17: days from the earliest quote in the purchase''s retrieval chain (within 90 days) to the purchase; 0 without one.', '', '', 'Input'),
        ('CustomerMonthPurchase', 'LP_Returned7d', 'Curation #17: 1 when LP_ReturnGapDays is 7 or more.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'MotorRenewals48m', 'Motor renewals completed by the month-end in the last 48 months.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_DueDate', 'Due date of the last Motor renewal completed by the month-end (due plus 30 days on or before it).', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_Renewals', 'Completed Motor renewals among the last three.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_ProactivityShare', 'Share of the last three renewals where the customer acted before the invitation (null below two observed).', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_Concessions', 'Fee concessions over the last three renewals.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_ConcessionsAfterContact', 'Fee concessions that followed a call from the customer, over the last three renewals (curation #21).', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_RequoteShare', 'Share of the last three renewals with an online re-quote (null below two covered).', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_RetainedShare', 'Share of the last three renewals retained (null below two).', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_AvgPromptsBeforeAction', 'Average prompts before the first customer action over the last three renewals.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'L3R_DistinctRoutes', 'Distinct renewal routes over the last three renewals (null below two observed).', '', '', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_3m', 'Renewal prompts Chill sent: pre-renewal notice, invitations, renewal CRM, alternative offer, lapse CRM. Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_12m', 'Renewal prompts Chill sent: pre-renewal notice, invitations, renewal CRM, alternative offer, lapse CRM. Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_24m', 'Renewal prompts Chill sent: pre-renewal notice, invitations, renewal CRM, alternative offer, lapse CRM. Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'ren_cust_actions_3m', 'Customer actions in a renewal: portal log-in, phones the call centre, renews online, on an inbound call or on chat. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'ren_cust_actions_12m', 'Customer actions in a renewal: portal log-in, phones the call centre, renews online, on an inbound call or on chat. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'ren_cust_actions_24m', 'Customer actions in a renewal: portal log-in, phones the call centre, renews online, on an inbound call or on chat. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_3m', 'Customer-initiated in-life contacts: change requests, document requests, looks for a change online. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_12m', 'Customer-initiated in-life contacts: change requests, document requests, looks for a change online. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_24m', 'Customer-initiated in-life contacts: change requests, document requests, looks for a change online. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_3m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_12m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_24m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_3m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_12m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_24m', 'Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_friction_events_3m', 'MyChill document-workflow friction (from Feb 2024): chases before submission, documents not submitted, cert not arrived. Last 3 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'doc_friction_events_12m', 'MyChill document-workflow friction (from Feb 2024): chases before submission, documents not submitted, cert not arrived. Last 12 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'doc_friction_events_24m', 'MyChill document-workflow friction (from Feb 2024): chases before submission, documents not submitted, cert not arrived. Last 24 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_3m', 'Document chase reminders and escalations sent by Chill. Last 3 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_12m', 'Document chase reminders and escalations sent by Chill. Last 12 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_24m', 'Document chase reminders and escalations sent by Chill. Last 24 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_3m', 'Document chase reminders and escalations sent by Chill. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_12m', 'Document chase reminders and escalations sent by Chill. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_24m', 'Document chase reminders and escalations sent by Chill. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_3m', 'Motor renewals retained with the fee reduced from the offered fee (TYEURFees below RenEURFee on PolicyRenewal), at the renewal sale date. Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_12m', 'Motor renewals retained with the fee reduced from the offered fee (TYEURFees below RenEURFee on PolicyRenewal), at the renewal sale date. Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_24m', 'Motor renewals retained with the fee reduced from the offered fee (TYEURFees below RenEURFee on PolicyRenewal), at the renewal sale date. Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_3m', 'Failed payment events in acquisition, renewal and MTA journeys. Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_12m', 'Failed payment events in acquisition, renewal and MTA journeys. Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_24m', 'Failed payment events in acquisition, renewal and MTA journeys. Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_il_3m', 'Failed payment events in acquisition, renewal and MTA journeys. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_il_12m', 'Failed payment events in acquisition, renewal and MTA journeys. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_il_24m', 'Failed payment events in acquisition, renewal and MTA journeys. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_logins_3m', 'Failed renewal portal log-ins (R2.F1). Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_logins_12m', 'Failed renewal portal log-ins (R2.F1). Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_logins_24m', 'Failed renewal portal log-ins (R2.F1). Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'arrears_events_3m', 'Arrears steps: missed payment, arrears chase, loan default. Last 3 months.', 'arrears', 'beh.cov_arrears', 'Input'),
        ('CustomerMonthBehaviour', 'arrears_events_12m', 'Arrears steps: missed payment, arrears chase, loan default. Last 12 months.', 'arrears', 'beh.cov_arrears', 'Input'),
        ('CustomerMonthBehaviour', 'arrears_events_24m', 'Arrears steps: missed payment, arrears chase, loan default. Last 24 months.', 'arrears', 'beh.cov_arrears', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_3m', 'Customer document requests (D1, HD1, VD1). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_12m', 'Customer document requests (D1, HD1, VD1). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_24m', 'Customer document requests (D1, HD1, VD1). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_3m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_12m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_24m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_3m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_12m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_24m', 'Inbound sales calls straight into the call centre (A0, HA0, VA0): phone quoting. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'events_total_3m', 'All linked episode events. Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'events_total_12m', 'All linked episode events. Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'events_total_24m', 'All linked episode events. Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_3m', 'Online Motor quotes started (A1). Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_12m', 'Online Motor quotes started (A1). Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_24m', 'Online Motor quotes started (A1). Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_il_3m', 'Online Motor quotes started (A1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_il_12m', 'Online Motor quotes started (A1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_il_24m', 'Online Motor quotes started (A1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_3m', 'Online Home quotes started (HA1). Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_12m', 'Online Home quotes started (HA1). Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_24m', 'Online Home quotes started (HA1). Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_il_3m', 'Online Home quotes started (HA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_il_12m', 'Online Home quotes started (HA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_il_24m', 'Online Home quotes started (HA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_3m', 'Online Van quotes started (VA1). Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_12m', 'Online Van quotes started (VA1). Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_24m', 'Online Van quotes started (VA1). Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_il_3m', 'Online Van quotes started (VA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_il_12m', 'Online Van quotes started (VA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_van_il_24m', 'Online Van quotes started (VA1). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'renewal_requotes_3m', 'Online re-quotes started from the renewal price (R2.B1.1, HR2.B1.1, VR2.B1.1). Last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'renewal_requotes_12m', 'Online re-quotes started from the renewal price (R2.B1.1, HR2.B1.1, VR2.B1.1). Last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'renewal_requotes_24m', 'Online re-quotes started from the renewal price (R2.B1.1, HR2.B1.1, VR2.B1.1). Last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_3m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_12m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_24m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_il_3m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_il_12m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_il_24m', 'Episodes, counted in the month of their last event (when the episode''s friction and outcome are known). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_3m', 'Episodes that contain at least one friction event (counted at their last event). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_12m', 'Episodes that contain at least one friction event (counted at their last event). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_24m', 'Episodes that contain at least one friction event (counted at their last event). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_il_3m', 'Episodes that contain at least one friction event (counted at their last event). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_il_12m', 'Episodes that contain at least one friction event (counted at their last event). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_friction_il_24m', 'Episodes that contain at least one friction event (counted at their last event). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_completed_3m', 'Episodes that reached an outcome step (counted at their last event). Last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_completed_12m', 'Episodes that reached an outcome step (counted at their last event). Last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_completed_24m', 'Episodes that reached an outcome step (counted at their last event). Last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_3m', 'Inbound voice conversations, QD_ queues excluded. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_12m', 'Inbound voice conversations, QD_ queues excluded. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_24m', 'Inbound voice conversations, QD_ queues excluded. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_3m', 'Inbound voice conversations, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_12m', 'Inbound voice conversations, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_24m', 'Inbound voice conversations, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_answered_3m', 'Inbound conversations answered by an agent. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_answered_12m', 'Inbound conversations answered by an agent. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_answered_24m', 'Inbound conversations answered by an agent. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandoned_3m', 'Inbound conversations abandoned. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandoned_12m', 'Inbound conversations abandoned. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandoned_24m', 'Inbound conversations abandoned. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_3m', 'Inbound conversations abandoned or waiting over 600 seconds. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_12m', 'Inbound conversations abandoned or waiting over 600 seconds. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_24m', 'Inbound conversations abandoned or waiting over 600 seconds. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_wait_s_3m', 'Seconds waited in queue on inbound conversations. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_wait_s_12m', 'Seconds waited in queue on inbound conversations. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_wait_s_24m', 'Seconds waited in queue on inbound conversations. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_talk_s_3m', 'Agent talk seconds on inbound conversations. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_talk_s_12m', 'Agent talk seconds on inbound conversations. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_talk_s_24m', 'Agent talk seconds on inbound conversations. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_3m', 'Outbound voice conversations to the customer, QD_ queues excluded. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_12m', 'Outbound voice conversations to the customer, QD_ queues excluded. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_24m', 'Outbound voice conversations to the customer, QD_ queues excluded. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_il_3m', 'Outbound voice conversations to the customer, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_il_12m', 'Outbound voice conversations to the customer, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_il_24m', 'Outbound voice conversations to the customer, QD_ queues excluded. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_talk_s_3m', 'Agent talk seconds on outbound conversations. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_talk_s_12m', 'Agent talk seconds on outbound conversations. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_out_talk_s_24m', 'Agent talk seconds on outbound conversations. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_3m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. Last 3 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_12m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. Last 12 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_24m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. Last 24 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_3m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_12m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_24m', 'MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mta_records_3m', 'MTA records in PolicyMta (history types P and T): the base for the change-type shares. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_records_12m', 'MTA records in PolicyMta (history types P and T): the base for the change-type shares. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_records_24m', 'MTA records in PolicyMta (history types P and T): the base for the change-type shares. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_forced_3m', 'MTAs that change the vehicle, registration or address. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_forced_12m', 'MTAs that change the vehicle, registration or address. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_forced_24m', 'MTAs that change the vehicle, registration or address. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_elective_3m', 'MTAs that change cover, excess, NCD, drivers, use or add-ons. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_elective_12m', 'MTAs that change cover, excess, NCD, drivers, use or add-ons. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_elective_24m', 'MTAs that change cover, excess, NCD, drivers, use or add-ons. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_cover_3m', 'MTAs touching cover, excess or NCD. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_cover_12m', 'MTAs touching cover, excess or NCD. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_cover_24m', 'MTAs touching cover, excess or NCD. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_drivers_3m', 'MTAs touching drivers or the driving option. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_drivers_12m', 'MTAs touching drivers or the driving option. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_drivers_24m', 'MTAs touching drivers or the driving option. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_vehicle_3m', 'MTAs touching the vehicle or registration. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_vehicle_12m', 'MTAs touching the vehicle or registration. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_vehicle_24m', 'MTAs touching the vehicle or registration. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_address_3m', 'MTAs touching the address. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_address_12m', 'MTAs touching the address. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_address_24m', 'MTAs touching the address. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_licence_3m', 'MTAs touching licence or convictions. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_licence_12m', 'MTAs touching licence or convictions. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_licence_24m', 'MTAs touching licence or convictions. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_use_3m', 'MTAs touching class or area of use. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_use_12m', 'MTAs touching class or area of use. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_use_24m', 'MTAs touching class or area of use. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_personal_3m', 'MTAs touching policyholder details. Last 3 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_personal_12m', 'MTAs touching policyholder details. Last 12 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_cat_personal_24m', 'MTAs touching policyholder details. Last 24 months.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'loan_status_changes_3m', 'Finance loan status changes (the opening status is not a change). Last 3 months.', 'loan', 'beh.cov_loan', 'Input'),
        ('CustomerMonthBehaviour', 'loan_status_changes_12m', 'Finance loan status changes (the opening status is not a change). Last 12 months.', 'loan', 'beh.cov_loan', 'Input'),
        ('CustomerMonthBehaviour', 'loan_status_changes_24m', 'Finance loan status changes (the opening status is not a change). Last 24 months.', 'loan', 'beh.cov_loan', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_3m', 'CRM messages sent (campaign history rows with a send). Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_12m', 'CRM messages sent (campaign history rows with a send). Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_24m', 'CRM messages sent (campaign history rows with a send). Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_il_3m', 'CRM messages sent (campaign history rows with a send). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_il_12m', 'CRM messages sent (campaign history rows with a send). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_sent_il_24m', 'CRM messages sent (campaign history rows with a send). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_3m', 'Sent messages opened at least once (a flag per message, not the open count). Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_12m', 'Sent messages opened at least once (a flag per message, not the open count). Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_24m', 'Sent messages opened at least once (a flag per message, not the open count). Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_il_3m', 'Sent messages opened at least once (a flag per message, not the open count). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_il_12m', 'Sent messages opened at least once (a flag per message, not the open count). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_opened_il_24m', 'Sent messages opened at least once (a flag per message, not the open count). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_3m', 'Sent messages clicked at least once. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_12m', 'Sent messages clicked at least once. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_24m', 'Sent messages clicked at least once. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_il_3m', 'Sent messages clicked at least once. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_il_12m', 'Sent messages clicked at least once. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_clicked_il_24m', 'Sent messages clicked at least once. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_bounced_3m', 'Sent messages that bounced. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_bounced_12m', 'Sent messages that bounced. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'msgs_bounced_24m', 'Sent messages that bounced. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_assisted_3m', 'Episodes with an agent step (phone, chat, web assist) or a linked voice call between their first and last event (#25). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_assisted_12m', 'Episodes with an agent step (phone, chat, web assist) or a linked voice call between their first and last event (#25). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_assisted_24m', 'Episodes with an agent step (phone, chat, web assist) or a linked voice call between their first and last event (#25). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_channel_tagged_3m', 'Episodes with at least one online, phone or chat step whose time is actual (the base for channel switches, #26). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_channel_tagged_12m', 'Episodes with at least one online, phone or chat step whose time is actual (the base for channel switches, #26). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_channel_tagged_24m', 'Episodes with at least one online, phone or chat step whose time is actual (the base for channel switches, #26). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_3m', 'Changes of channel (online, phone, chat) between consecutive time-usable steps within episodes (#26). A lower bound: derived-time steps are left out. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_12m', 'Changes of channel (online, phone, chat) between consecutive time-usable steps within episodes (#26). A lower bound: derived-time steps are left out. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_24m', 'Changes of channel (online, phone, chat) between consecutive time-usable steps within episodes (#26). A lower bound: derived-time steps are left out. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_repeat_contact_3m', 'Episodes with two or more linked inbound calls between their first and last event (#43). Same issue is inferred from the episode only: resolution steps are not built. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_repeat_contact_12m', 'Episodes with two or more linked inbound calls between their first and last event (#43). Same issue is inferred from the episode only: resolution steps are not built. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episodes_repeat_contact_24m', 'Episodes with two or more linked inbound calls between their first and last event (#43). Same issue is inferred from the episode only: resolution steps are not built. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_3m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_12m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_24m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_il_3m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_il_12m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_il_24m', 'New-business Motor, Home and Van sales (PolicySale, first sale of the policy, rebooked MTAs excluded). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_3m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_12m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_24m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_il_3m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_il_12m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'nb_sales_web_assist_il_24m', 'New-business sales through Web Assist: started online, completed with an agent (PolicySale.Channel ''b) Web Assist'', #27). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'doc_decisions_3m', 'MyChill document decisions (accepted or rejected), one per workflow record (#46). Last 3 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_decisions_12m', 'MyChill document decisions (accepted or rejected), one per workflow record (#46). Last 12 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_decisions_24m', 'MyChill document decisions (accepted or rejected), one per workflow record (#46). Last 24 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejections_3m', 'MyChill documents rejected (#46). Last 3 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejections_12m', 'MyChill documents rejected (#46). Last 12 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejections_24m', 'MyChill documents rejected (#46). Last 24 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_3m', 'Upload attempts on decided MyChill documents (sum of Attempts, #46). Last 3 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_12m', 'Upload attempts on decided MyChill documents (sum of Attempts, #46). Last 12 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_24m', 'Upload attempts on decided MyChill documents (sum of Attempts, #46). Last 24 months.', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_3m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_12m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_24m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_il_3m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_il_12m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_il_24m', 'Documents not received: repeat document-request calls (D3.F1, HD3.F1, VD3.F1) and replacement certificates (A7.B1.5, HA7.B1.5, R6.B1.5, HR6.B1.5, VR6.B1.5) (#53). In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_premium_3m', 'Gross premium (EUR, at the previous month-end) of the policies behind doc_nonreceipt events, once per policy per month (#53). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_premium_12m', 'Gross premium (EUR, at the previous month-end) of the policies behind doc_nonreceipt events, once per policy per month (#53). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_nonreceipt_premium_24m', 'Gross premium (EUR, at the previous month-end) of the policies behind doc_nonreceipt events, once per policy per month (#53). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_3m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. Last 3 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_12m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. Last 12 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_24m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. Last 24 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_il_3m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_il_12m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_actions_il_24m', 'Marketing consent statements recorded through the contact-centre interfaces on any channel (Mercury, ActionId 1 opt-in or 2 opt-out, 3 and 4 excluded as the warehouse does). Quote-form statements are left out: logged only in 2022 and 2023. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_3m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. Last 3 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_12m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. Last 12 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_24m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. Last 24 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_il_3m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 3 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_il_12m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 12 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawals_il_24m', 'Marketing consent withdrawn: an opt-out after an opt-in on the same contact and channel, recorded through the contact-centre interfaces (#64). The consent log grows by 40% in 2023, so compare customers within a period, not across years. In-life only: not in a renewal episode, or (without an episode) outside the customer''s renewal windows. Last 24 months.', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_quotes_3m', 'MFQ online Motor quotes linked to the customer. Before May 2026 almost only the quotes on the way to an online purchase (quotes that did not lead to a sale do not link), and those count from the sale date, when the link became known, not the quote date. From May 2026 a quote counts on its own date. Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_quotes_12m', 'MFQ online Motor quotes linked to the customer. Before May 2026 almost only the quotes on the way to an online purchase (quotes that did not lead to a sale do not link), and those count from the sale date, when the link became known, not the quote date. From May 2026 a quote counts on its own date. Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_quotes_24m', 'MFQ online Motor quotes linked to the customer. Before May 2026 almost only the quotes on the way to an online purchase (quotes that did not lead to a sale do not link), and those count from the sale date, when the link became known, not the quote date. From May 2026 a quote counts on its own date. Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_retrievals_3m', 'Linked quotes that retrieved an earlier saved quote, counted at the retrieving quote (curation #8: the warehouse RetrievalCount is read from today''s row, so it is not used). Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_retrievals_12m', 'Linked quotes that retrieved an earlier saved quote, counted at the retrieving quote (curation #8: the warehouse RetrievalCount is read from today''s row, so it is not used). Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_retrievals_24m', 'Linked quotes that retrieved an earlier saved quote, counted at the retrieving quote (curation #8: the warehouse RetrievalCount is read from today''s row, so it is not used). Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_insurers_3m', 'Insurers priced on those quotes (sum of InsurerCount, curation #10). Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_insurers_12m', 'Insurers priced on those quotes (sum of InsurerCount, curation #10). Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_insurers_24m', 'Insurers priced on those quotes (sum of InsurerCount, curation #10). Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_returned_3m', 'Linked quotes that went back to a saved quote 7 days or more old, counted at the returning quote (curation #17, read along the retrieval chain: from 2023 a retrieval opens a new quote, so a quote''s own activity no longer spans the gap). Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_returned_12m', 'Linked quotes that went back to a saved quote 7 days or more old, counted at the returning quote (curation #17, read along the retrieval chain: from 2023 a retrieval opens a new quote, so a quote''s own activity no longer spans the gap). Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_returned_24m', 'Linked quotes that went back to a saved quote 7 days or more old, counted at the returning quote (curation #17, read along the retrieval chain: from 2023 a retrieval opens a new quote, so a quote''s own activity no longer spans the gap). Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_compared_3m', 'Quotes with a chosen cover (Comprehensive or TPFT, from the selection steps) that follow an earlier such quote of the customer (curation #11). Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_compared_12m', 'Quotes with a chosen cover (Comprehensive or TPFT, from the selection steps) that follow an earlier such quote of the customer (curation #11). Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_compared_24m', 'Quotes with a chosen cover (Comprehensive or TPFT, from the selection steps) that follow an earlier such quote of the customer (curation #11). Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_down_3m', 'Of those, quotes moving from Comprehensive to TPFT. Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_down_12m', 'Of those, quotes moving from Comprehensive to TPFT. Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_down_24m', 'Of those, quotes moving from Comprehensive to TPFT. Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_up_3m', 'Of those, quotes moving from TPFT to Comprehensive. Last 3 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_up_12m', 'Of those, quotes moving from TPFT to Comprehensive. Last 12 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_cover_up_24m', 'Of those, quotes moving from TPFT to Comprehensive. Last 24 months.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_tagged_3m', 'Linked quotes with a first and a last device recorded (curation #28). Last 3 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_tagged_12m', 'Linked quotes with a first and a last device recorded (curation #28). Last 12 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_tagged_24m', 'Linked quotes with a first and a last device recorded (curation #28). Last 24 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_switch_3m', 'Of those, quotes whose last device differs from the first. Last 3 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_switch_12m', 'Of those, quotes whose last device differs from the first. Last 12 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'mfq_device_switch_24m', 'Of those, quotes whose last device differs from the first. Last 24 months.', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_3m', 'Web sessions linked to the customer while they held a policy (#54, #57, #74). A customer whose visits cannot be linked reads 0: the GA cookie links only on a device that quoted online, or through the email on an MFQ quote. Last 3 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_12m', 'Web sessions linked to the customer while they held a policy (#54, #57, #74). A customer whose visits cannot be linked reads 0: the GA cookie links only on a device that quoted online, or through the email on an MFQ quote. Last 12 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_24m', 'Web sessions linked to the customer while they held a policy (#54, #57, #74). A customer whose visits cannot be linked reads 0: the GA cookie links only on a device that quoted online, or through the email on an MFQ quote. Last 24 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_nonquote_3m', 'Linked web sessions with no Motor or Home quote reference and no MFQ quote (#54). Last 3 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_nonquote_12m', 'Linked web sessions with no Motor or Home quote reference and no MFQ quote (#54). Last 12 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_sessions_nonquote_24m', 'Linked web sessions with no Motor or Home quote reference and no MFQ quote (#54). Last 24 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_page_views_3m', 'Page views in linked web sessions. Last 3 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_page_views_12m', 'Page views in linked web sessions. Last 12 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_page_views_24m', 'Page views in linked web sessions. Last 24 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_content_views_3m', 'Page views of content pages in linked sessions: Chill''s page groups Chill Blog, Chill Cust Care, Chill Company and Insurers (#57). Last 3 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_content_views_12m', 'Page views of content pages in linked sessions: Chill''s page groups Chill Blog, Chill Cust Care, Chill Company and Insurers (#57). Last 12 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_content_views_24m', 'Page views of content pages in linked sessions: Chill''s page groups Chill Blog, Chill Cust Care, Chill Company and Insurers (#57). Last 24 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_other_product_sessions_3m', 'Linked sessions with pages of a product the customer did not hold at the month-end of the session: Home, Van or Motor not held, or Travel, Life, Gadget or Commercial (not tracked as holdings, so always counted) (#74). Last 3 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_other_product_sessions_12m', 'Linked sessions with pages of a product the customer did not hold at the month-end of the session: Home, Van or Motor not held, or Travel, Life, Gadget or Commercial (not tracked as holdings, so always counted) (#74). Last 12 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'web_other_product_sessions_24m', 'Linked sessions with pages of a product the customer did not hold at the month-end of the session: Home, Van or Motor not held, or Travel, Life, Gadget or Commercial (not tracked as holdings, so always counted) (#74). Last 24 months.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'calls_other_queues_3m', 'Inbound calls to queues outside the Motor renewal base''s own product (beh_codes.MOTOR_BASE_OTHER_QUEUES: Home, Van and Life sales, non-standard Motor, Other, CS_Other general service, Chat - Customer Service) with no sale, renewal or MTA for the customer within 7 days after the call, counted 7 days after the call when that is known (#56). Last 3 months.', 'tcall', 'beh.cov_tcall', 'Input'),
        ('CustomerMonthBehaviour', 'calls_other_queues_12m', 'Inbound calls to queues outside the Motor renewal base''s own product (beh_codes.MOTOR_BASE_OTHER_QUEUES: Home, Van and Life sales, non-standard Motor, Other, CS_Other general service, Chat - Customer Service) with no sale, renewal or MTA for the customer within 7 days after the call, counted 7 days after the call when that is known (#56). Last 12 months.', 'tcall', 'beh.cov_tcall', 'Input'),
        ('CustomerMonthBehaviour', 'calls_other_queues_24m', 'Inbound calls to queues outside the Motor renewal base''s own product (beh_codes.MOTOR_BASE_OTHER_QUEUES: Home, Van and Life sales, non-standard Motor, Other, CS_Other general service, Chat - Customer Service) with no sale, renewal or MTA for the customer within 7 days after the call, counted 7 days after the call when that is known (#56). Last 24 months.', 'tcall', 'beh.cov_tcall', 'Input'),
        ('CustomerMonthBehaviour', 'friction_all_3m', 'Friction events, every code including the codes first seen after Nov 2022 and Chill process issues, counted on their date (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_all_12m', 'Friction events, every code including the codes first seen after Nov 2022 and Chill process issues, counted on their date (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_all_24m', 'Friction events, every code including the codes first seen after Nov 2022 and Chill process issues, counted on their date (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_3m', 'Friction events followed by a later main-journey (spine) step with a real time in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_12m', 'Friction events followed by a later main-journey (spine) step with a real time in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_24m', 'Friction events followed by a later main-journey (spine) step with a real time in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_3m', 'Friction events of the payment family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_12m', 'Friction events of the payment family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_24m', 'Friction events of the payment family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_3m', 'Friction events of the payment family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_12m', 'Friction events of the payment family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_24m', 'Friction events of the payment family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_3m', 'Friction events of the login family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_12m', 'Friction events of the login family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_24m', 'Friction events of the login family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_3m', 'Friction events of the login family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_12m', 'Friction events of the login family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_24m', 'Friction events of the login family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_3m', 'Friction events of the contact family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_12m', 'Friction events of the contact family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_24m', 'Friction events of the contact family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_3m', 'Friction events of the contact family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_12m', 'Friction events of the contact family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_24m', 'Friction events of the contact family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_3m', 'Friction events of the documents family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_12m', 'Friction events of the documents family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_24m', 'Friction events of the documents family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_3m', 'Friction events of the documents family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_12m', 'Friction events of the documents family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_24m', 'Friction events of the documents family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_3m', 'Friction events of the quote family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_12m', 'Friction events of the quote family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_24m', 'Friction events of the quote family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_3m', 'Friction events of the quote family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_12m', 'Friction events of the quote family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_24m', 'Friction events of the quote family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_3m', 'Friction events of the process family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_12m', 'Friction events of the process family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_24m', 'Friction events of the process family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_3m', 'Friction events of the process family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_12m', 'Friction events of the process family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_24m', 'Friction events of the process family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_3m', 'Friction events of the other family (beh_codes.FRICTION_FAMILY), every code (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_12m', 'Friction events of the other family (beh_codes.FRICTION_FAMILY), every code (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_24m', 'Friction events of the other family (beh_codes.FRICTION_FAMILY), every code (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_3m', 'Friction events of the other family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_12m', 'Friction events of the other family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_24m', 'Friction events of the other family followed by a later main-journey step in the same episode, counted on the date of that step (#49). Last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'offers_sent_3m', 'Offer campaigns sent to the customer: promotions and offers, renewal price offers and winbacks Chill confirmed (build/offer_campaigns.py, #61). Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offers_sent_12m', 'Offer campaigns sent to the customer: promotions and offers, renewal price offers and winbacks Chill confirmed (build/offer_campaigns.py, #61). Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offers_sent_24m', 'Offer campaigns sent to the customer: promotions and offers, renewal price offers and winbacks Chill confirmed (build/offer_campaigns.py, #61). Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offers_clicked_3m', 'Offer campaigns the customer clicked, counted on the send date. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offers_clicked_12m', 'Offer campaigns the customer clicked, counted on the send date. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offers_clicked_24m', 'Offer campaigns the customer clicked, counted on the send date. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_nb_3m', 'Clicked offer campaigns followed by a new-business sale within 30 days of the click, counted on the sale date (#61). Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_nb_12m', 'Clicked offer campaigns followed by a new-business sale within 30 days of the click, counted on the sale date (#61). Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_nb_24m', 'Clicked offer campaigns followed by a new-business sale within 30 days of the click, counted on the sale date (#61). Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_renewal_3m', 'Clicked offer campaigns followed by a renewal within 30 days of the click, counted on the renewal sale date (#61). Renewal price offers go to customers already renewing, so this is mostly the renewal that was due. Last 3 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_renewal_12m', 'Clicked offer campaigns followed by a renewal within 30 days of the click, counted on the renewal sale date (#61). Renewal price offers go to customers already renewing, so this is mostly the renewal that was due. Last 12 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_clicks_renewal_24m', 'Clicked offer campaigns followed by a renewal within 30 days of the click, counted on the renewal sale date (#61). Renewal price offers go to customers already renewing, so this is mostly the renewal that was due. Last 24 months.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_eligible_3m', 'Friction measures of spec v3.3 the customer was eligible for (had the opportunity to meet), summed over their episodes that ended in the window: Car, Home and Van, first year, renewal, change and document-request episodes (beh_codes.FRICTION_V3_MEASURES). Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_eligible_12m', 'Friction measures of spec v3.3 the customer was eligible for (had the opportunity to meet), summed over their episodes that ended in the window: Car, Home and Van, first year, renewal, change and document-request episodes (beh_codes.FRICTION_V3_MEASURES). Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_eligible_24m', 'Friction measures of spec v3.3 the customer was eligible for (had the opportunity to meet), summed over their episodes that ended in the window: Car, Home and Van, first year, renewal, change and document-request episodes (beh_codes.FRICTION_V3_MEASURES). Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_affected_3m', 'Of those, the measures where the customer met the friction (spec v3.3 Affected = 1). Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_affected_12m', 'Of those, the measures where the customer met the friction (spec v3.3 Affected = 1). Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_affected_24m', 'Of those, the measures where the customer met the friction (spec v3.3 Affected = 1). Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_res_known_3m', 'Affected measures with a known resolution (spec v3.3 Resolution Resolved or Unresolved; Unknown and measures the spec gives no resolution are left out). Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_res_known_12m', 'Affected measures with a known resolution (spec v3.3 Resolution Resolved or Unresolved; Unknown and measures the spec gives no resolution are left out). Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_res_known_24m', 'Affected measures with a known resolution (spec v3.3 Resolution Resolved or Unresolved; Unknown and measures the spec gives no resolution are left out). Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_3m', 'Affected measures resolved (spec v3.3 Resolution = Resolved). Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_12m', 'Affected measures resolved (spec v3.3 Resolution = Resolved). Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_24m', 'Affected measures resolved (spec v3.3 Resolution = Resolved). Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_eligible_3m', 'As friction_v3_eligible, renewal episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_eligible_12m', 'As friction_v3_eligible, renewal episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_eligible_24m', 'As friction_v3_eligible, renewal episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_affected_3m', 'As friction_v3_affected, renewal episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_affected_12m', 'As friction_v3_affected, renewal episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_affected_24m', 'As friction_v3_affected, renewal episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_res_known_3m', 'As friction_v3_res_known, renewal episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_res_known_12m', 'As friction_v3_res_known, renewal episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_res_known_24m', 'As friction_v3_res_known, renewal episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_3m', 'As friction_v3_resolved, renewal episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_12m', 'As friction_v3_resolved, renewal episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_24m', 'As friction_v3_resolved, renewal episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_eligible_3m', 'As friction_v3_eligible, first year episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_eligible_12m', 'As friction_v3_eligible, first year episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_eligible_24m', 'As friction_v3_eligible, first year episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_affected_3m', 'As friction_v3_affected, first year episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_affected_12m', 'As friction_v3_affected, first year episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_affected_24m', 'As friction_v3_affected, first year episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_res_known_3m', 'As friction_v3_res_known, first year episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_res_known_12m', 'As friction_v3_res_known, first year episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_res_known_24m', 'As friction_v3_res_known, first year episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_3m', 'As friction_v3_resolved, first year episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_12m', 'As friction_v3_resolved, first year episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_24m', 'As friction_v3_resolved, first year episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_eligible_3m', 'As friction_v3_eligible, change episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_eligible_12m', 'As friction_v3_eligible, change episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_eligible_24m', 'As friction_v3_eligible, change episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_affected_3m', 'As friction_v3_affected, change episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_affected_12m', 'As friction_v3_affected, change episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_affected_24m', 'As friction_v3_affected, change episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_res_known_3m', 'As friction_v3_res_known, change episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_res_known_12m', 'As friction_v3_res_known, change episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_res_known_24m', 'As friction_v3_res_known, change episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_3m', 'As friction_v3_resolved, change episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_12m', 'As friction_v3_resolved, change episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_24m', 'As friction_v3_resolved, change episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_eligible_3m', 'As friction_v3_eligible, document request episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_eligible_12m', 'As friction_v3_eligible, document request episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_eligible_24m', 'As friction_v3_eligible, document request episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_affected_3m', 'As friction_v3_affected, document request episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_affected_12m', 'As friction_v3_affected, document request episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_affected_24m', 'As friction_v3_affected, document request episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_res_known_3m', 'As friction_v3_res_known, document request episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_res_known_12m', 'As friction_v3_res_known, document request episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_res_known_24m', 'As friction_v3_res_known, document request episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_3m', 'As friction_v3_resolved, document request episodes only. Last 3 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_12m', 'As friction_v3_resolved, document request episodes only. Last 12 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_24m', 'As friction_v3_resolved, document request episodes only. Last 24 months.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'months_held_3m', 'Month-ends in the window at which the customer held an in-force Motor, Home or Van policy. Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'months_held_12m', 'Month-ends in the window at which the customer held an in-force Motor, Home or Van policy. Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'months_held_24m', 'Month-ends in the window at which the customer held an in-force Motor, Home or Van policy. Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'renewals_due_3m', 'Offered Motor, Home and Van renewals due in the window (opportunity for renewal behaviour). Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'renewals_due_12m', 'Offered Motor, Home and Van renewals due in the window (opportunity for renewal behaviour). Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'renewals_due_24m', 'Offered Motor, Home and Van renewals due in the window (opportunity for renewal behaviour). Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'motor_renewals_due_3m', 'Offered Motor renewals due in the window. Last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'motor_renewals_due_12m', 'Offered Motor renewals due in the window. Last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'motor_renewals_due_24m', 'Offered Motor renewals due in the window. Last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_pm_3m', 'calls_in per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_pm_12m', 'calls_in per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_pm_24m', 'calls_in per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_recent_vs_prior', 'calls_in per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_pm_3m', 'calls_in_il per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_pm_12m', 'calls_in_il per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_pm_24m', 'calls_in_il per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_il_recent_vs_prior', 'calls_in_il per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_pm_3m', 'inlife_contacts per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_pm_12m', 'inlife_contacts per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_pm_24m', 'inlife_contacts per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'inlife_contacts_recent_vs_prior', 'inlife_contacts per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts. Renewal-driven at M11: non-renewers ask for documents in the months before renewal, so the gap between renewers and non-renewers widens at M11 (not leakage: the activity is before the month-end).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_pm_3m', 'friction_events per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_pm_12m', 'friction_events per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_pm_24m', 'friction_events per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_recent_vs_prior', 'friction_events per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_pm_3m', 'friction_events_il per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_pm_12m', 'friction_events_il per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_pm_24m', 'friction_events_il per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_events_il_recent_vs_prior', 'friction_events_il per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_pm_3m', 'mtas per month held, last 3 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_pm_12m', 'mtas per month held, last 12 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_pm_24m', 'mtas per month held, last 24 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_recent_vs_prior', 'mtas per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_pm_3m', 'mtas_il per month held, last 3 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_pm_12m', 'mtas_il per month held, last 12 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_pm_24m', 'mtas_il per month held, last 24 months.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'mtas_il_recent_vs_prior', 'mtas_il per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'mta_count', 'beh.cov_mta_count', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_pm_3m', 'doc_requests per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_pm_12m', 'doc_requests per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_pm_24m', 'doc_requests per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'doc_requests_recent_vs_prior', 'doc_requests per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts. Renewal-driven at M11: non-renewers ask for documents in the months before renewal, so the gap between renewers and non-renewers widens at M11 (not leakage: the activity is before the month-end).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'chases_pm_3m', 'chases per month held, last 3 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_pm_12m', 'chases per month held, last 12 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_pm_24m', 'chases per month held, last 24 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_recent_vs_prior', 'chases per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_pm_3m', 'chases_il per month held, last 3 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_pm_12m', 'chases_il per month held, last 12 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_pm_24m', 'chases_il per month held, last 24 months.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'chases_il_recent_vs_prior', 'chases_il per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'mychill', 'beh.cov_mychill', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_pm_3m', 'failed_payments per month held, last 3 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_pm_12m', 'failed_payments per month held, last 12 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_pm_24m', 'failed_payments per month held, last 24 months.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'failed_payments_recent_vs_prior', 'failed_payments per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_pm_3m', 'sales_calls per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_pm_12m', 'sales_calls per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_pm_24m', 'sales_calls per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_recent_vs_prior', 'sales_calls per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_pm_3m', 'sales_calls_il per month held, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_pm_12m', 'sales_calls_il per month held, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_pm_24m', 'sales_calls_il per month held, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'sales_calls_il_recent_vs_prior', 'sales_calls_il per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_pm_3m', 'online_quotes_motor per month held, last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_pm_12m', 'online_quotes_motor per month held, last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_pm_24m', 'online_quotes_motor per month held, last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_motor_recent_vs_prior', 'online_quotes_motor per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_pm_3m', 'online_quotes_home per month held, last 3 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_pm_12m', 'online_quotes_home per month held, last 12 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_pm_24m', 'online_quotes_home per month held, last 24 months.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'online_quotes_home_recent_vs_prior', 'online_quotes_home per month held in the last 3 months minus the same rate over months 4 to 24. Positive = more recently than usual. Null without 24 months of coverage or without months held in both parts.', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'had_renewal_3m', '1 when an offered renewal fell due in the last 3 months: the opportunity for renewal behaviour.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'had_comms_3m', '1 when Chill sent at least one CRM message in the last 3 months: the opportunity to engage.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_per_renewal_3m', 'Renewal prompts per renewal due, last 3 months. Null without a renewal.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'ren_actions_per_renewal_3m', 'Customer renewal actions per renewal due, last 3 months. Null without a renewal.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_per_motor_renewal_3m', 'In-flight concessions per Motor renewal due, last 3 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'requotes_per_renewal_3m', 'Online re-quotes from the renewal price per renewal due, last 3 months (quote coverage applies).', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_3m', 'Share of CRM messages sent in the last 3 months that were opened. Null without a send.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_3m', 'Share of CRM messages sent in the last 3 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'bounce_rate_3m', 'Share of CRM messages sent in the last 3 months that bounced (contactability).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'mta_breadth_3m', 'Number of distinct MTA change categories used in the last 3 months (0 to 7).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_elective_share_3m', 'Share of MTAs in the last 3 months that were elective (cover, excess, NCD, drivers, use). Null without an MTA.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_forced_share_3m', 'Share of MTAs in the last 3 months that were forced (vehicle, registration, address).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'friction_episode_share_3m', 'Share of episodes started in the last 3 months that contain friction.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episode_completion_rate_3m', 'Share of episodes started in the last 3 months that reached an outcome step (many outcome steps are not built: gap 11).', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandon_rate_3m', 'Share of inbound calls in the last 3 months that were abandoned.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_rate_3m', 'Share of inbound calls in the last 3 months abandoned or waiting over 600 seconds.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_wait_s_3m', 'Average queue wait in seconds on inbound calls, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_talk_s_3m', 'Average agent talk seconds on inbound calls, last 3 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_il_3m', 'Share of CRM messages sent outside the renewal windows in the last 3 months that were opened.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_il_3m', 'Share of CRM messages sent outside the renewal windows in the last 3 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'assisted_episode_share_3m', 'Share of episodes in the last 3 months with an agent step or a linked call (curation #25). Null without an episode.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_per_episode_3m', 'Channel switches per episode with a channel-tagged step, last 3 months (curation #26, a lower bound).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'repeat_contact_episode_share_3m', 'Share of episodes in the last 3 months with two or more linked inbound calls (curation #43).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'web_assist_rate_3m', 'Share of new-business sales in the last 3 months made through Web Assist (curation #27). Null without a sale.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejection_rate_3m', 'Share of MyChill document decisions in the last 3 months that were rejections (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_per_doc_3m', 'Upload attempts per decided MyChill document, last 3 months (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'quote_retrieval_share_3m', 'Share of linked MFQ quotes in the last 3 months that retrieved an earlier saved quote (curation #8). Null without a linked quote.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'insurers_per_quote_3m', 'Average insurers priced per linked MFQ quote, last 3 months (curation #10).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'quote_return_share_3m', 'Share of linked MFQ quotes in the last 3 months that went back to a saved quote 7 days or more old (curation #17).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_downgrade_share_3m', 'Share of compared quotes in the last 3 months that moved from Comprehensive to TPFT against the previous quote (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_drift_net_3m', 'Upgrades minus downgrades per compared quote, last 3 months: -1 all down, +1 all up, 0 no change (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'device_consistent_3m', '1 when every linked MFQ quote in the last 3 months ended on the device it started on, 0 when any switched, null without a device-tagged quote (curation #28).', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'content_page_share_3m', 'Share of page views in the customer''s linked web sessions of the last 3 months that were content pages (Chill Blog, Chill Cust Care, Chill Company, Insurers; curation #57). Null without a linked session.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'other_product_viewed_3m', '1 when a linked web session in the last 3 months viewed pages of a product the customer did not hold, 0 when none did, null without a linked session (curation #74).', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_share_3m', 'Friction events resolved in the last 3 months (a later main-journey step in the same episode, counted when it happened) per friction event in the same months, capped at 1 (curation #49). Null without friction. Friction steps resolved 80% of the time or more add little difference between customers (out/run_note_simon.md section 11).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'offer_click_rate_3m', 'Share of offer campaigns sent in the last 3 months that the customer clicked (curation #61). Null without an offer.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_nb_3m', 'Clicked offers followed by a new-business sale within 30 days, per offer sent, last 3 months (curation #61).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_renewal_3m', 'Clicked offers followed by a renewal within 30 days, per offer sent, last 3 months (curation #61). Mostly renewal price offers to customers already renewing.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawn_3m', '1 when the customer withdrew marketing consent on any channel in the last 3 months (curation #64).', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_share_3m', 'As friction_resolved_share_3m for the payment friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_share_3m', 'As friction_resolved_share_3m for the login friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_share_3m', 'As friction_resolved_share_3m for the contact friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_share_3m', 'As friction_resolved_share_3m for the documents friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_share_3m', 'As friction_resolved_share_3m for the quote friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_share_3m', 'As friction_resolved_share_3m for the process friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_share_3m', 'As friction_resolved_share_3m for the other friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_rate_3m', 'Friction rate to spec v3.3 in the episodes that ended in the last 3 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_share_3m', 'Share of the customer''s spec v3.3 friction in the episodes that ended in the last 3 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_rate_3m', 'Friction rate to spec v3.3 in the renewal episodes that ended in the last 3 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_share_3m', 'Share of the customer''s spec v3.3 friction in the renewal episodes that ended in the last 3 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_rate_3m', 'Friction rate to spec v3.3 in the first-year episodes that ended in the last 3 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_share_3m', 'Share of the customer''s spec v3.3 friction in the first-year episodes that ended in the last 3 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_rate_3m', 'Friction rate to spec v3.3 in the change episodes that ended in the last 3 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_share_3m', 'Share of the customer''s spec v3.3 friction in the change episodes that ended in the last 3 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_rate_3m', 'Friction rate to spec v3.3 in the document-request episodes that ended in the last 3 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_share_3m', 'Share of the customer''s spec v3.3 friction in the document-request episodes that ended in the last 3 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'had_renewal_12m', '1 when an offered renewal fell due in the last 12 months: the opportunity for renewal behaviour.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'had_comms_12m', '1 when Chill sent at least one CRM message in the last 12 months: the opportunity to engage.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_per_renewal_12m', 'Renewal prompts per renewal due, last 12 months. Null without a renewal.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'ren_actions_per_renewal_12m', 'Customer renewal actions per renewal due, last 12 months. Null without a renewal.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_per_motor_renewal_12m', 'In-flight concessions per Motor renewal due, last 12 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'requotes_per_renewal_12m', 'Online re-quotes from the renewal price per renewal due, last 12 months (quote coverage applies).', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_12m', 'Share of CRM messages sent in the last 12 months that were opened. Null without a send.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_12m', 'Share of CRM messages sent in the last 12 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'bounce_rate_12m', 'Share of CRM messages sent in the last 12 months that bounced (contactability).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'mta_breadth_12m', 'Number of distinct MTA change categories used in the last 12 months (0 to 7).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_elective_share_12m', 'Share of MTAs in the last 12 months that were elective (cover, excess, NCD, drivers, use). Null without an MTA.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_forced_share_12m', 'Share of MTAs in the last 12 months that were forced (vehicle, registration, address).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'friction_episode_share_12m', 'Share of episodes started in the last 12 months that contain friction.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episode_completion_rate_12m', 'Share of episodes started in the last 12 months that reached an outcome step (many outcome steps are not built: gap 11).', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandon_rate_12m', 'Share of inbound calls in the last 12 months that were abandoned.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_rate_12m', 'Share of inbound calls in the last 12 months abandoned or waiting over 600 seconds.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_wait_s_12m', 'Average queue wait in seconds on inbound calls, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_talk_s_12m', 'Average agent talk seconds on inbound calls, last 12 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_il_12m', 'Share of CRM messages sent outside the renewal windows in the last 12 months that were opened.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_il_12m', 'Share of CRM messages sent outside the renewal windows in the last 12 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'assisted_episode_share_12m', 'Share of episodes in the last 12 months with an agent step or a linked call (curation #25). Null without an episode.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_per_episode_12m', 'Channel switches per episode with a channel-tagged step, last 12 months (curation #26, a lower bound).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'repeat_contact_episode_share_12m', 'Share of episodes in the last 12 months with two or more linked inbound calls (curation #43).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'web_assist_rate_12m', 'Share of new-business sales in the last 12 months made through Web Assist (curation #27). Null without a sale.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejection_rate_12m', 'Share of MyChill document decisions in the last 12 months that were rejections (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_per_doc_12m', 'Upload attempts per decided MyChill document, last 12 months (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'quote_retrieval_share_12m', 'Share of linked MFQ quotes in the last 12 months that retrieved an earlier saved quote (curation #8). Null without a linked quote.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'insurers_per_quote_12m', 'Average insurers priced per linked MFQ quote, last 12 months (curation #10).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'quote_return_share_12m', 'Share of linked MFQ quotes in the last 12 months that went back to a saved quote 7 days or more old (curation #17).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_downgrade_share_12m', 'Share of compared quotes in the last 12 months that moved from Comprehensive to TPFT against the previous quote (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_drift_net_12m', 'Upgrades minus downgrades per compared quote, last 12 months: -1 all down, +1 all up, 0 no change (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'device_consistent_12m', '1 when every linked MFQ quote in the last 12 months ended on the device it started on, 0 when any switched, null without a device-tagged quote (curation #28).', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'content_page_share_12m', 'Share of page views in the customer''s linked web sessions of the last 12 months that were content pages (Chill Blog, Chill Cust Care, Chill Company, Insurers; curation #57). Null without a linked session.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'other_product_viewed_12m', '1 when a linked web session in the last 12 months viewed pages of a product the customer did not hold, 0 when none did, null without a linked session (curation #74).', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_share_12m', 'Friction events resolved in the last 12 months (a later main-journey step in the same episode, counted when it happened) per friction event in the same months, capped at 1 (curation #49). Null without friction. Friction steps resolved 80% of the time or more add little difference between customers (out/run_note_simon.md section 11).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'offer_click_rate_12m', 'Share of offer campaigns sent in the last 12 months that the customer clicked (curation #61). Null without an offer.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_nb_12m', 'Clicked offers followed by a new-business sale within 30 days, per offer sent, last 12 months (curation #61).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_renewal_12m', 'Clicked offers followed by a renewal within 30 days, per offer sent, last 12 months (curation #61). Mostly renewal price offers to customers already renewing.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawn_12m', '1 when the customer withdrew marketing consent on any channel in the last 12 months (curation #64).', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_share_12m', 'As friction_resolved_share_12m for the payment friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_share_12m', 'As friction_resolved_share_12m for the login friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_share_12m', 'As friction_resolved_share_12m for the contact friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_share_12m', 'As friction_resolved_share_12m for the documents friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_share_12m', 'As friction_resolved_share_12m for the quote friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_share_12m', 'As friction_resolved_share_12m for the process friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_share_12m', 'As friction_resolved_share_12m for the other friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_rate_12m', 'Friction rate to spec v3.3 in the episodes that ended in the last 12 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_share_12m', 'Share of the customer''s spec v3.3 friction in the episodes that ended in the last 12 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_rate_12m', 'Friction rate to spec v3.3 in the renewal episodes that ended in the last 12 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_share_12m', 'Share of the customer''s spec v3.3 friction in the renewal episodes that ended in the last 12 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_rate_12m', 'Friction rate to spec v3.3 in the first-year episodes that ended in the last 12 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_share_12m', 'Share of the customer''s spec v3.3 friction in the first-year episodes that ended in the last 12 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_rate_12m', 'Friction rate to spec v3.3 in the change episodes that ended in the last 12 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_share_12m', 'Share of the customer''s spec v3.3 friction in the change episodes that ended in the last 12 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_rate_12m', 'Friction rate to spec v3.3 in the document-request episodes that ended in the last 12 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_share_12m', 'Share of the customer''s spec v3.3 friction in the document-request episodes that ended in the last 12 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'had_renewal_24m', '1 when an offered renewal fell due in the last 24 months: the opportunity for renewal behaviour.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'had_comms_24m', '1 when Chill sent at least one CRM message in the last 24 months: the opportunity to engage.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'prompts_per_renewal_24m', 'Renewal prompts per renewal due, last 24 months. Null without a renewal.', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'ren_actions_per_renewal_24m', 'Customer renewal actions per renewal due, last 24 months. Null without a renewal.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'concessions_per_motor_renewal_24m', 'In-flight concessions per Motor renewal due, last 24 months.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'requotes_per_renewal_24m', 'Online re-quotes from the renewal price per renewal due, last 24 months (quote coverage applies).', 'quotes', 'beh.cov_quotes', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_24m', 'Share of CRM messages sent in the last 24 months that were opened. Null without a send.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_24m', 'Share of CRM messages sent in the last 24 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'bounce_rate_24m', 'Share of CRM messages sent in the last 24 months that bounced (contactability).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'mta_breadth_24m', 'Number of distinct MTA change categories used in the last 24 months (0 to 7).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_elective_share_24m', 'Share of MTAs in the last 24 months that were elective (cover, excess, NCD, drivers, use). Null without an MTA.', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'mta_forced_share_24m', 'Share of MTAs in the last 24 months that were forced (vehicle, registration, address).', 'mta', 'beh.cov_mta', 'Input'),
        ('CustomerMonthBehaviour', 'friction_episode_share_24m', 'Share of episodes started in the last 24 months that contain friction.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'episode_completion_rate_24m', 'Share of episodes started in the last 24 months that reached an outcome step (many outcome steps are not built: gap 11).', 'events', 'beh.cov_events', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_abandon_rate_24m', 'Share of inbound calls in the last 24 months that were abandoned.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_long_wait_rate_24m', 'Share of inbound calls in the last 24 months abandoned or waiting over 600 seconds.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_wait_s_24m', 'Average queue wait in seconds on inbound calls, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'calls_in_avg_talk_s_24m', 'Average agent talk seconds on inbound calls, last 24 months.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_il_24m', 'Share of CRM messages sent outside the renewal windows in the last 24 months that were opened.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_il_24m', 'Share of CRM messages sent outside the renewal windows in the last 24 months that were clicked.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'assisted_episode_share_24m', 'Share of episodes in the last 24 months with an agent step or a linked call (curation #25). Null without an episode.', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'channel_switches_per_episode_24m', 'Channel switches per episode with a channel-tagged step, last 24 months (curation #26, a lower bound).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'repeat_contact_episode_share_24m', 'Share of episodes in the last 24 months with two or more linked inbound calls (curation #43).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'web_assist_rate_24m', 'Share of new-business sales in the last 24 months made through Web Assist (curation #27). Null without a sale.', 'spine', 'beh.hist_start', 'Input'),
        ('CustomerMonthBehaviour', 'doc_rejection_rate_24m', 'Share of MyChill document decisions in the last 24 months that were rejections (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'doc_attempts_per_doc_24m', 'Upload attempts per decided MyChill document, last 24 months (curation #46).', 'docwf', 'beh.cov_docwf', 'Input'),
        ('CustomerMonthBehaviour', 'quote_retrieval_share_24m', 'Share of linked MFQ quotes in the last 24 months that retrieved an earlier saved quote (curation #8). Null without a linked quote.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'insurers_per_quote_24m', 'Average insurers priced per linked MFQ quote, last 24 months (curation #10).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'quote_return_share_24m', 'Share of linked MFQ quotes in the last 24 months that went back to a saved quote 7 days or more old (curation #17).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_downgrade_share_24m', 'Share of compared quotes in the last 24 months that moved from Comprehensive to TPFT against the previous quote (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'cover_drift_net_24m', 'Upgrades minus downgrades per compared quote, last 24 months: -1 all down, +1 all up, 0 no change (curation #11).', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthBehaviour', 'device_consistent_24m', '1 when every linked MFQ quote in the last 24 months ended on the device it started on, 0 when any switched, null without a device-tagged quote (curation #28).', 'mfq_device', 'beh.cov_mfq_device', 'Input'),
        ('CustomerMonthBehaviour', 'content_page_share_24m', 'Share of page views in the customer''s linked web sessions of the last 24 months that were content pages (Chill Blog, Chill Cust Care, Chill Company, Insurers; curation #57). Null without a linked session.', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'other_product_viewed_24m', '1 when a linked web session in the last 24 months viewed pages of a product the customer did not hold, 0 when none did, null without a linked session (curation #74).', 'web', 'beh.cov_web', 'Input'),
        ('CustomerMonthBehaviour', 'friction_resolved_share_24m', 'Friction events resolved in the last 24 months (a later main-journey step in the same episode, counted when it happened) per friction event in the same months, capped at 1 (curation #49). Null without friction. Friction steps resolved 80% of the time or more add little difference between customers (out/run_note_simon.md section 11).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'offer_click_rate_24m', 'Share of offer campaigns sent in the last 24 months that the customer clicked (curation #61). Null without an offer.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_nb_24m', 'Clicked offers followed by a new-business sale within 30 days, per offer sent, last 24 months (curation #61).', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'offer_response_renewal_24m', 'Clicked offers followed by a renewal within 30 days, per offer sent, last 24 months (curation #61). Mostly renewal price offers to customers already renewing.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'consent_withdrawn_24m', '1 when the customer withdrew marketing consent on any channel in the last 24 months (curation #64).', 'consent', 'beh.cov_consent', 'Input'),
        ('CustomerMonthBehaviour', 'friction_payment_resolved_share_24m', 'As friction_resolved_share_24m for the payment friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_login_resolved_share_24m', 'As friction_resolved_share_24m for the login friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_contact_resolved_share_24m', 'As friction_resolved_share_24m for the contact friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_documents_resolved_share_24m', 'As friction_resolved_share_24m for the documents friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_quote_resolved_share_24m', 'As friction_resolved_share_24m for the quote friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_process_resolved_share_24m', 'As friction_resolved_share_24m for the process friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_other_resolved_share_24m', 'As friction_resolved_share_24m for the other friction family (beh_codes.FRICTION_FAMILY).', 'calls', 'beh.cov_calls', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_rate_24m', 'Friction rate to spec v3.3 in the episodes that ended in the last 24 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_resolved_share_24m', 'Share of the customer''s spec v3.3 friction in the episodes that ended in the last 24 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_rate_24m', 'Friction rate to spec v3.3 in the renewal episodes that ended in the last 24 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_renewal_resolved_share_24m', 'Share of the customer''s spec v3.3 friction in the renewal episodes that ended in the last 24 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_rate_24m', 'Friction rate to spec v3.3 in the first-year episodes that ended in the last 24 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_firstyear_resolved_share_24m', 'Share of the customer''s spec v3.3 friction in the first-year episodes that ended in the last 24 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_rate_24m', 'Friction rate to spec v3.3 in the change episodes that ended in the last 24 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_change_resolved_share_24m', 'Share of the customer''s spec v3.3 friction in the change episodes that ended in the last 24 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_rate_24m', 'Friction rate to spec v3.3 in the document-request episodes that ended in the last 24 months: measures where the customer met the friction per measure they were eligible for (Affected / Eligible). Null without an eligible measure (no opportunity), which is different from 0 (opportunity, no friction).', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'friction_v3_docreq_resolved_share_24m', 'Share of the customer''s spec v3.3 friction in the document-request episodes that ended in the last 24 months that was resolved, of the affected measures with a known resolution. Null without one.', 'fv3', 'beh.cov_friction_v3', 'Input'),
        ('CustomerMonthBehaviour', 'open_rate_recent_vs_prior', 'open_rate on messages sent in the last 3 months minus open_rate on messages sent in months 4 to 24.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'click_rate_recent_vs_prior', 'click_rate on messages sent in the last 3 months minus click_rate on messages sent in months 4 to 24.', 'comms', 'beh.cov_comms', 'Input'),
        ('CustomerMonthBehaviour', 'failed_login_unrecovered_24m', '1 when the customer''s last failed renewal-portal log-in in the last 24 months has no successful log-in after it by the month-end, 0 when it has, null without a failed log-in (curation #52; the workbook''s event names and join would find nothing, so the source is read directly). Source break: from March 2025 the portal logs about twice as many failed log-ins for the same number of successful ones, and stops setting IsLocked in August 2025, so readings before and after early 2025 are not comparable.', 'portal', 'beh.cov_portal', 'Input'),
        ('CustomerMonthBehaviour', 'excess_elasticity_24m', 'Range of voluntary excess across the customer''s linked MFQ quotes in the last 24 months, as a proportion of their most common excess (curation #12). Null below two quotes with an excess, or when the most common excess is 0.', 'mfq', 'beh.cov_mfq', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_Retained', 'Retained on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_RenewalRoute', 'RenewalRoute on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_PromptsBeforeAction', 'PromptsBeforeAction on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_ActedBeforeInvite', 'ActedBeforeInvite on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_RenewedOnOfferNoContact', 'RenewedOnOfferNoContact on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_DaysInviteToAction', 'DaysInviteToAction on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_ReassuranceCall', 'ReassuranceCall on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_Concession', 'Concession on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_ConcessionAfterContact', 'ConcessionAfterContact on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_Requote', 'Requote on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_PriceMovement', 'PriceMovement on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_PriceRise5', 'PriceRise5 on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_RenewedAfterPriceRise', 'RenewedAfterPriceRise on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_PaidVsOffered', 'PaidVsOffered on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_GWPYoYCustPct', 'GWPYoYCustPct on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_PriceIncreasePC', 'PriceIncreasePC on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_FrictionEvents', 'FrictionEvents on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_FailedPayments', 'FailedPayments on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_FailedLogins', 'FailedLogins on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_InboundCalls', 'InboundCalls on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_ProcessingSubChannel', 'ProcessingSubChannel on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_FrictionV3Rate', 'FrictionV3Rate on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthRenewalHistory', 'LR_FrictionV3ResolvedShare', 'FrictionV3ResolvedShare on the last Motor renewal completed by the month-end.', '', '', 'Input'),
        ('CustomerMonthOutcome', 'c2s_calls_in_next3m', 'OUTCOME. Inbound voice conversations, QD_ queues excluded. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_in_next12m', 'OUTCOME. Inbound voice conversations, QD_ queues excluded. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_in_talk_s_next3m', 'OUTCOME. Agent talk seconds on inbound conversations. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_in_talk_s_next12m', 'OUTCOME. Agent talk seconds on inbound conversations. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_out_next3m', 'OUTCOME. Outbound voice conversations to the customer, QD_ queues excluded. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_out_next12m', 'OUTCOME. Outbound voice conversations to the customer, QD_ queues excluded. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_out_talk_s_next3m', 'OUTCOME. Agent talk seconds on outbound conversations. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_calls_out_talk_s_next12m', 'OUTCOME. Agent talk seconds on outbound conversations. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_inlife_contacts_next3m', 'OUTCOME. Customer-initiated in-life contacts: change requests, document requests, looks for a change online. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_inlife_contacts_next12m', 'OUTCOME. Customer-initiated in-life contacts: change requests, document requests, looks for a change online. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_mtas_next3m', 'OUTCOME. MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_mtas_next12m', 'OUTCOME. MTAs, the warehouse count (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1), dated on the posting day. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_doc_requests_next3m', 'OUTCOME. Customer document requests (D1, HD1, VD1). Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_doc_requests_next12m', 'OUTCOME. Customer document requests (D1, HD1, VD1). Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_chases_next3m', 'OUTCOME. Document chase reminders and escalations sent by Chill. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_chases_next12m', 'OUTCOME. Document chase reminders and escalations sent by Chill. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_failed_payments_next3m', 'OUTCOME. Failed payment events in acquisition, renewal and MTA journeys. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_failed_payments_next12m', 'OUTCOME. Failed payment events in acquisition, renewal and MTA journeys. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_friction_events_next3m', 'OUTCOME. Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_friction_events_next12m', 'OUTCOME. Friction events on the codes present since Nov 2022 (stable set; later codes excluded, see beh_codes.FRICTION_LATE). Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_arrears_events_next3m', 'OUTCOME. Arrears steps: missed payment, arrears chase, loan default. Next 3 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'c2s_arrears_events_next12m', 'OUTCOME. Arrears steps: missed payment, arrears chase, loan default. Next 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellObservable3m', '1 when the 3-month cross-sell window ends by outcome_end, so the XSell 3m columns are set.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'XSellMotor3m', 'OUTCOME. 1 when a new-business Motor policy was bought within 3 months, 0 when not. Null when the customer held Motor at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellHome3m', 'OUTCOME. 1 when a new-business Home policy was bought within 3 months, 0 when not. Null when the customer held Home at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellVan3m', 'OUTCOME. 1 when a new-business Van policy was bought within 3 months, 0 when not. Null when the customer held Van at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellAny3m', 'OUTCOME. 1 when any product not held at the month-end was bought within 3 months. Null when all three were held or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellObservable6m', '1 when the 6-month cross-sell window ends by outcome_end, so the XSell 6m columns are set.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'XSellMotor6m', 'OUTCOME. 1 when a new-business Motor policy was bought within 6 months, 0 when not. Null when the customer held Motor at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellHome6m', 'OUTCOME. 1 when a new-business Home policy was bought within 6 months, 0 when not. Null when the customer held Home at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellVan6m', 'OUTCOME. 1 when a new-business Van policy was bought within 6 months, 0 when not. Null when the customer held Van at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellAny6m', 'OUTCOME. 1 when any product not held at the month-end was bought within 6 months. Null when all three were held or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellObservable12m', '1 when the 12-month cross-sell window ends by outcome_end, so the XSell 12m columns are set.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'XSellMotor12m', 'OUTCOME. 1 when a new-business Motor policy was bought within 12 months, 0 when not. Null when the customer held Motor at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellHome12m', 'OUTCOME. 1 when a new-business Home policy was bought within 12 months, 0 when not. Null when the customer held Home at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellVan12m', 'OUTCOME. 1 when a new-business Van policy was bought within 12 months, 0 when not. Null when the customer held Van at the month-end or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'XSellAny12m', 'OUTCOME. 1 when any product not held at the month-end was bought within 12 months. Null when all three were held or the window is not observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'C2SObservable3m', '1 when the next 3 months end by last_me, so the c2s 3m columns are set.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'C2SObservable12m', '1 when the next 12 months end by last_me, so the c2s 12m columns are set.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'FirstXSellDate12m', 'OUTCOME. Date of the first cross-sell within 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'FirstXSellProduct12m', 'OUTCOME. Product of the first cross-sell within 12 months.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'OutcomeMotorRenewalDate', 'OUTCOME. Due date of the nearest upcoming Motor renewal the next-renewal outcome refers to.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'NextMotorRenewalOffered', 'OUTCOME. RenewalOffered of the nearest upcoming Motor renewal.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'NextMotorRenewalRetained', 'OUTCOME. RenewalRetained of the nearest upcoming Motor renewal, set when its outcome is observable.', '', '', 'Outcome'),
        ('CustomerMonthOutcome', 'NextMotorRenewalObservable', '1 when the nearest upcoming Motor renewal is due by renewal_outcome_end.', '', '', 'Filter'),
        ('CustomerMonthOutcome', 'NextMotorRenewalSubChannel', 'OUTCOME. Channel the nearest upcoming Motor renewal was processed through.', '', '', 'Outcome')
        as t(TableName, ColumnName, Description, CoverageFamily, CoverageSetting, Role)


## Dictionary completeness


In [ ]:
%%pyspark
# Every column Simon reads must have a description and a role: a column without one could be an outcome used as an
# input. Fails the run when a column of the analysis tables is missing from the dictionary.
described = {r.ColumnName for r in spark.table(f"{target_schema}.Dictionary").select("ColumnName").collect()}
missing = {t: sorted(set(spark.table(f"{target_schema}.{t}").columns) - described)
           for t in ["CustomerMonthDataset", "MotorPolicyMonthDataset", "MotorRenewalCycle"]}
missing = {t: c for t, c in missing.items() if c}
if missing:
    raise Exception(f"Columns without a dictionary entry (add them to build/nb60_dataset.py DICT): {missing}")
print("Dictionary covers every column of the analysis tables.")


## First read


In [ ]:
%%pyspark
# A first read of the build: rows and coverage by year, for the run note. Nothing is written.
spark.sql(f"""
    select  year(MonthEndDate)                              as Year,
            count(*)                                        as CustomerMonths,
            count(distinct SCV_Customer_Key)                as Customers,
            avg(calls_in_12m)                               as AvgCallsIn12m,
            avg(case when calls_in_12m is null then 1.0 else 0.0 end) as ShareCallsNull,
            avg(case when online_quotes_motor_12m is null then 1.0 else 0.0 end) as ShareQuotesNull,
            avg(XSellAny12m)                                as XSellAny12mRate,
            avg(NextMotorRenewalRetained)                   as NextMotorRetainedRate
    from    {target_schema}.CustomerMonthDataset
    where   IsAnalysisMonth = 1
    group by year(MonthEndDate)
    order by 1""").show(20, truncate=False)


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_60_Dataset", [{"table": "{target}.Coverage", "key": ["CoverageFamily"]}, {"table": "{target}.Dictionary", "key": ["TableName", "ColumnName"]}, {"table": "{target}.CustomerMonthDataset", "key": ["SCV_Customer_Key", "MonthEndDate"]}, {"table": "{target}.MotorPolicyMonthDataset", "key": ["PolicyCode", "MonthEndDate"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_60_Dataset", "succeeded")
